# Load package

In [1]:
library(dplyr)
library(Seurat)
library(patchwork)    
library(data.table)
library(parallel)
library(matrixStats)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loading required package: SeuratObject

Loading required package: sp


Attaching package: ‘SeuratObject’


The following objects are masked from ‘package:base’:

    intersect, t


Warning message:
“package ‘data.table’ was built under R version 4.4.3”

Attaching package: ‘data.table’


The following objects are masked from ‘package:dplyr’:

    between, first, last


Warning message:
“package ‘matrixStats’ was built under R version 4.4.2”

Attaching package: ‘matrixStats’


The following object is masked from ‘package:dplyr’:

    count




In [2]:
sessionInfo()

R version 4.4.1 (2024-06-14)
Platform: x86_64-conda-linux-gnu
Running under: CentOS Linux 7 (Core)

Matrix products: default
BLAS/LAPACK: /hwfssz3/PS_JLU/zhangxiao6/software/miniconda3/envs/R4.41/lib/libopenblasp-r0.3.29.so;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=en_US.UTF-8       LC_NUMERIC=C              
 [3] LC_TIME=en_US.UTF-8        LC_COLLATE=en_US.UTF-8    
 [5] LC_MONETARY=en_US.UTF-8    LC_MESSAGES=en_US.UTF-8   
 [7] LC_PAPER=en_US.UTF-8       LC_NAME=C                 
 [9] LC_ADDRESS=C               LC_TELEPHONE=C            
[11] LC_MEASUREMENT=en_US.UTF-8 LC_IDENTIFICATION=C       

time zone: Asia/Shanghai
tzcode source: system (glibc)

attached base packages:
[1] parallel  stats     graphics  grDevices utils     datasets  methods  
[8] base     

other attached packages:
[1] matrixStats_1.5.0  data.table_1.17.2  patchwork_1.3.0    Seurat_5.3.0      
[5] SeuratObject_5.1.0 sp_2.2-0           dplyr_1.1.4       

loaded via a namespace (and not attached):
  [1] del

# Data merging and processing

## Set working directory: contains CellChat result files for each brain region

In [4]:
setwd("~/path/to/your/directory/")

In [ ]:
custom_lr_path <- "~/interaction_input_CellChatDB_update_20250909_LGY_modfiy_V1.csv"
interaction_df <- read.csv(custom_lr_path, row.names = 1)
complex_ref <- readRDS("~/CellChatDB.human.complex.gene_list.rds")

In [8]:
head(complex_ref,2)

,gene_name,complex_name
,<chr>,<chr>
Activin AB,INHBA,Activin AB
Inhibin A,INHA,Inhibin A


## Define a function to merge data across different brain regions

In [7]:
merge_cellchat_precise_v2 <- function(input_dir, output_dir,
                                      regions = c("PFC", "Hf", "Str", "Tha", "Hyt", "MB", "PM", "Cer"),
                                      age_groups = c("Young", "Middle_age", "Old", "Exceptionally_old"),
                                      interaction_df = interaction_df, 
                                      complex_ref = complex_ref,
                                      n_threads = 144) {
    # Define function to process a single region
    .process_region <- function(region) {
        message("\nProcessing region: ", region, "------", format(Sys.time(), "%Y-%m-%d %H:%M:%S"))
        # 1. Read all age group files for this region
        age_data <- list()
        for (age in age_groups) {
            file_pattern <- paste0(region, "_", age, "_subsetCommunication_prob_truncatedMean.csv")
            file_path <- file.path(input_dir, file_pattern)
      
            if (file.exists(file_path)) {
                df <- fread(file_path, data.table = F, nThread = n_threads)
                
                df$annotation <- interaction_df[match(df$interaction_name, interaction_df$interaction_name), "annotation"]
                message("Read raw1 ", nrow(df), " interactions from ", age)
                df <- df[!is.na(df$annotation),]
                
                df$interaction_name_2 <- interaction_df[match(df$interaction_name_2, interaction_df$interaction_name_2), "annotation"]
                message("Read raw2 ", nrow(df), " interactions from ", age)
                df <- df[!is.na(df$annotation),]

                df$pathway_name <- interaction_df[match(df$interaction_name, interaction_df$interaction_name), "pathway_name"]

                #拆分complex_LR_gene
                inter_df <- df
                tmp = merge(inter_df, complex_ref, by.x = 3, by.y = 2, all.x = T)
                inter_df = merge(tmp, complex_ref, by.x = 4, by.y = 2, all.x = T)
                inter_df$gene_name.x[is.na(inter_df$gene_name.x)] = inter_df$ligand[is.na(inter_df$gene_name.x)]
                inter_df$gene_name.y[is.na(inter_df$gene_name.y)] = inter_df$receptor[is.na(inter_df$gene_name.y)]
                colnames(inter_df)[c(12,13)] = c("ligand_gene","receptor_gene")
                df <- inter_df

                df$region <- region
                df$age_group <- gsub("_", " ", age)
                
                # Create unique interaction ID (without splitting)
                df$interaction_id <- paste0(df$source, "#", 
                                            df$target, "#", 
                                            df$ligand, "#", 
                                            df$receptor, "#", 
                                            df$interaction_name, "#", 
                                            df$interaction_name_2, "#", 
                                            df$pathway_name, "#", 
                                            df$annotation, "#", 
                                            df$region, "#", 
                                            df$ligand_gene, "#", 
                                            df$receptor_gene)
                df %>% distinct(interaction_id, .keep_all =TRUE) -> dfun #去除重复变量
                rownames(dfun) <- dfun$interaction_id

                age_data[[age]] <- dfun
                message("Get uncomplex ", nrow(dfun), " interactions from ", age)
            } else {
                warning("File not found: ", file_path)
                age_data[[age]] <- NULL
            }
        }

        #2. Merge all age groups and find unique interactions
        merged_data <- Reduce(rbind, age_data)
        unique_interactions <- unique(merged_data$interaction_id)
        message("\nTotal unique interactions across all age groups: ", length(unique_interactions))

        # 3. Create complete template with all unique interactions
        merged_data %>% distinct(interaction_id, .keep_all =TRUE) -> template
        if(dim(template)[1] == length(unique_interactions)) {
            template$prob  <- 0
            template$pval  <- NA
            rownames(template) <- template$interaction_id
        } else {
            stop("unique_interactions number not match with the length of table!!!")
        } 

        # 4. Create complete dataset for each age group
        complete_age_data <- list()

        for(age in names(age_data)) {
            df_old <- age_data[[age]]
            df_new <- template
            df_new$age_group <- age
            df_new[rownames(df_old), "prob"] <- df_old$prob
            df_new[rownames(df_old), "pval"] <- df_old$pval
            message("Get all ", nrow(df_new), " interactions from ", age)
            complete_age_data[[age]] <- df_new
        }

        # 5. Combine all age groups for this region
        region_complete <- Reduce(rbind, complete_age_data)
        message("\nFinal interaction counts after completion:", dim(region_complete)[1], "------", format(Sys.time(), "%Y-%m-%d %H:%M:%S"))

        return(region_complete)
        }

    # Process all regions
   final_results <- list()
    for (region in regions) {
        region_result <- .process_region(region)
        if (!is.null(region_result)) {
            final_results[[region]] <- region_result
            
            # Save this region's results
            filename <- file.path(output_dir, paste0(region, "_merge_cellchat_truncatedMean_LR_pair_result.csv"))
            message("Saving results for ", region, " to: ", filename, "------", format(Sys.time(), "%Y-%m-%d %H:%M:%S"))
            fwrite(region_result, filename, nThread = n_threads)
        }
    }
    
    # Combine all regions
    message("\nCombining results from all regions...")
    all_regions_complete <- Reduce(rbind, final_results)
    message("Total unique interactions across all regions: ", dim(all_regions_complete)[1], "------", format(Sys.time(), "%Y-%m-%d %H:%M:%S"))
  
    # Save combined results
    combined_file1 <- file.path(output_dir, "01.NHPABC_CellChat_truncatedMean_merge.rds")
    message("\nWriting complete results for all regions to: ", combined_file1, "------", format(Sys.time(), "%Y-%m-%d %H:%M:%S"))
    saveRDS(all_regions_complete, combined_file1)

    combined_file2 <- file.path(output_dir, "01.NHPABC_CellChat_truncatedMean_merge.csv")
    message("\nWriting complete results for all regions to: ", combined_file2, "------", format(Sys.time(), "%Y-%m-%d %H:%M:%S"))
    fwrite(all_regions_complete, combined_file2, nThread = n_threads)

    result_back <- list(
        Each_region_results = final_results,
        all_regions_complete = all_regions_complete
    )

    return(result_back)
}

## Execute data merging (estimated time: 20~60 mins)

In [ ]:
merge_all <- merge_cellchat_precise_v2(
  input_dir = "~/01.Project/NHPABC/Figure5/03.Processed_CellChat_prob_truncatedMean/02.result",
  output_dir = "~/01.Project/NHPABC/Figure5/04.NHPABC_Aging_CCI/02.result/",
  regions = c("PFC", "Hf", "Str", "Tha", "Hyt", "MB", "PM", "Cer"),
  age_groups = c("Young", "Middle_age", "Old", "Exceptionally_old"),
  interaction_df = interaction_df,
  complex_ref = complex_ref,
  n_threads = 140
)


Processing region: PFC------2025-10-18 00:07:19

Read raw1 1819692 interactions from Young

Read raw2 1819692 interactions from Young

Get uncomplex 2518211 interactions from Young

Read raw1 1575461 interactions from Middle_age

Read raw2 1575461 interactions from Middle_age



In [9]:
merge_all <- fread("~/01.Project/NHPABC/Figure5/04.NHPABC_Aging_CCI/02.result/01.NHPABC_CellChat_truncatedMean_merge.csv", nThread =  140, data.table = F)

In [10]:
head(merge_all,2)
dim(merge_all)

,receptor,ligand,source,target,prob,pval,interaction_name,interaction_name_2,pathway_name,annotation,evidence,ligand_gene,receptor_gene,region,age_group,interaction_id
,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,ABCA1,APOA1,Ast1,Ast1,2.577457e-06,0.88,APOA1_ABCA1,Cell-Cell Contact,ApoA,Cell-Cell Contact,PMID: 20064972,APOA1,ABCA1,PFC,Young,Ast1#Ast1#APOA1#ABCA1#APOA1_ABCA1#Cell-Cell Contact#ApoA#Cell-Cell Contact#PFC#APOA1#ABCA1
2,ABCA1,APOA1,CGE CNR1+ Inh,Ast1,9.248340e-06,0.06,APOA1_ABCA1,Cell-Cell Contact,ApoA,Cell-Cell Contact,PMID: 20064972,APOA1,ABCA1,PFC,Young,CGE CNR1+ Inh#Ast1#APOA1#ABCA1#APOA1_ABCA1#Cell-Cell Contact#ApoA#Cell-Cell Contact#PFC#APOA1#ABCA1


[1] 189005488        16

## Modify merged data information

In [11]:
cci <- as.data.frame(merge_all$all_regions_complete,stringsAsFactors = FALSE)

In [12]:
cci <- merge_all

In [14]:
table(cci$region,cci$age_group)

     
      Exceptionally_old Middle_age     Old   Young
  Cer           2335351    2335351 2335351 2335351
  Hf            4089145    4089145 4089145 4089145
  Hyt           8805272    8805272 8805272 8805272
  MB            8500791    8500791 8500791 8500791
  PFC           3691010    3691010 3691010 3691010
  PM            8435489    8435489 8435489 8435489
  Str           3371808    3371808 3371808 3371808
  Tha           8022506    8022506 8022506 8022506

### Cell type grouping: define Source to Target (CellToCell)

In [16]:
sort(unique(c(cci$source, cci$target)))

[1] "Ast1"              "Ast2"              "Ast3"             
 [4] "AVP_N"             "BEC"               "CA1/SUB deep Ex"  
 [7] "CA1/SUB s.f. Ex"   "CA2/4 Ex"          "CALCR_N"          
[10] "CARTPT_N"          "CBLN2+ Ex"         "Cerebellar GC"    
[13] "CGE CNR1+ Inh"     "CGE LAMP5+ Inh"    "CHO_N"            
[16] "Choroid plexus"    "COP"               "CRH_N"            
[19] "CRHBP_N"           "DG Ex"             "EC L2 Ex"         
[22] "EC L3/5 Ex"        "EC L6 Ex"          "Ependymal"        
[25] "eSPN"              "Fibroblast"        "HCRT_N"           
[28] "HDC_N"             "ICj Inh"           "L2/3-IT Ex"       
[31] "L4/5-IT Ex"        "L5-ET Ex"          "L5-IT Ex"         
[34] "L5/6-NP Ex"        "L6-CT Ex"          "L6-IT Ex"         
[37] "Lower rhombic lip" "Macrophage"        "MB-derived Inh"   
[40] "MGE CRABP1+ Inh"   "MGE PVALB+ Inh"    "MGE SST+ Inh"     
[43] "MGE TAC3+ Inh"     "MGE UNC5B+ Inh"    "Mic1"             
[46] "Mic2"              "Mic3"              "MLI1"             
[49] "MLI2"              "ODC1"              "ODC2"             
[52] "OPC1"              "OPC2"              "OXT_N"            
[55] "PENK_N"            "Pericyte"          "PLI"              
[58] "PMCH_N"            "POMC_N"            "Purkinje"         
[61] "RYR3+ Ex"          "SLC24A2_N"         "SLC32A1_N"        
[64] "SMC"               "SPN-D1"            "SPN-D1-NUDAP"     
[67] "SPN-D2"            "SST_N"             "TAC1_N"           
[70] "TH_A10"            "TH_A9"             "TPH_N"            
[73] "TPH2_N"            "TRH_N"             "TRHR_N"           
[76] "UBC"               "Upper rhombic lip"

In [ ]:
#source_celltype, takes ~3 minutes
cci <- cci %>%
  mutate(source_celltype = case_when(
    grepl("Ast|Mic|ODC|OPC", source) ~ "Glia", 
    grepl("VS|Ependymal|Fibroblast|BEC|Choroid|Macrophage|Pericyte|SMC|UBC", source) ~ "Other",
    TRUE ~ "Neuron"
  ))

#target_celltype, takes ~3 minutes
cci <- cci %>%
  mutate(target_celltype = case_when(
    grepl("Ast|Mic|ODC|OPC", target) ~ "Glia",
    grepl("VS|Ependymal|Fibroblast|BEC|Choroid|Macrophage|Pericyte|SMC|UBC", target) ~ "Other",
    TRUE ~ "Neuron" 
  ))

In [19]:
cci$CellToCell = paste0(cci$source_celltype,"_",cci$target_celltype)

## Integrate data from different age groups
***
The core function of this code is to integrate and standardize cell-cell interaction (CCI) data across different age groups, facilitating subsequent analysis of differences in cell-cell communication across age stages.

### Construct unique identifiers (ID)
***
The number of unique identifiers (ID) should be one-fourth of the total number of entries in the cell-cell interaction (CCI) table.

In [21]:
inter_all <- cci

In [22]:
inter_all$ID <- paste0(inter_all$interaction_id,"#", inter_all$CellToCell)

In [23]:
unique_ID  = distinct(inter_all, ID, .keep_all = T)$ID

In [31]:
dim(inter_all)
dim(inter_all)[1]/4
length(unique(inter_all$ID))
length(unique_ID)

[1] 189005488        20

[1] 47251372

[1] 47251372

[1] 47251372

### Split the merged data by age group

In [34]:
df.list = list()
message("\nProcessing split age group data", "------", format(Sys.time(), "%Y-%m-%d %H:%M:%S"))
for(f in unique(inter_all$age_group)){
    message("\nProcessing ",f , " Start------", format(Sys.time(), "%Y-%m-%d %H:%M:%S"))
    df = subset(inter_all, age_group == f)[,c(3,4,1,2,12,13,7:10,14,15,16,19,20,5:6)]
    colnames(df)[c(16,17)] = c(f,paste0("pval_",f))
    rownames(df) = df$ID
    df = df[unique_ID,]
    df.list[[f]] = df
    message("\nProcessing ",f , " End------", format(Sys.time(), "%Y-%m-%d %H:%M:%S"))
}


Processing split age group data------2025-12-09 10:27:31


Processing Young Start------2025-12-09 10:27:37


Processing Young End------2025-12-09 10:32:11


Processing Middle_age Start------2025-12-09 10:32:11


Processing Middle_age End------2025-12-09 10:36:05


Processing Old Start------2025-12-09 10:36:05


Processing Old End------2025-12-09 10:38:26


Processing Exceptionally_old Start------2025-12-09 10:38:26


Processing Exceptionally_old End------2025-12-09 10:40:49



In [36]:
# Merge data in the order of Young→Middle age→Old→Exceptionally old
inter_single = cbind(df.list[[4]],
                     df.list[[2]][,16:17],
                     df.list[[3]][,16:17],
                     df.list[[1]][,16:17] )
inter_single <- inter_single[,c(1:15,16,18,20,22,17,19,21,23)]
rownames(inter_single) = NULL

In [39]:
head(inter_single,2)

,source,target,receptor,ligand,ligand_gene,receptor_gene,interaction_name,interaction_name_2,pathway_name,annotation,⋯,CellToCell,ID,Exceptionally_old,Middle_age,Old,Young,pval_Exceptionally_old,pval_Middle_age,pval_Old,pval_Young
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,Ast1,Ast1,ABCA1,APOA1,APOA1,ABCA1,APOA1_ABCA1,Cell-Cell Contact,ApoA,Cell-Cell Contact,⋯,Glia_Glia,Ast1#Ast1#APOA1#ABCA1#APOA1_ABCA1#Cell-Cell Contact#ApoA#Cell-Cell Contact#PFC#APOA1#ABCA1#Glia_Glia,0.000000e+00,8.832057e-06,7.808092e-06,2.577457e-06,NA,0.46,0.37,0.88
2,CGE CNR1+ Inh,Ast1,ABCA1,APOA1,APOA1,ABCA1,APOA1_ABCA1,Cell-Cell Contact,ApoA,Cell-Cell Contact,⋯,Neuron_Glia,CGE CNR1+ Inh#Ast1#APOA1#ABCA1#APOA1_ABCA1#Cell-Cell Contact#ApoA#Cell-Cell Contact#PFC#APOA1#ABCA1#Neuron_Glia,1.335173e-05,2.646694e-05,1.579015e-05,9.248340e-06,0.11,0.00,0.00,0.06


# Perform initial filtering on CCI data
***
Perform multi-step filtering on cell-cell interaction (CCI) data to screen for high-quality, biologically meaningful communication events

Filter standard：
1. which **pvalue > 0.05** will be deleted.
2. Delete those CCI with **NA** value in pvalue of prob.
3. Filter the low score CCI base on the mean prob of four age stage, and the **mean < 0.002** will be deleted.

In [43]:
#' Calculate and output sample size and filtering ratio before and after filtering
#' 
#' @param df_before Data frame before filtering
#' @param df_after Data frame after filtering
#' @param step_name Optional parameter to identify the current filtering step (e.g., "Step1"), default is "Filtering"
#' 
#' @return No return value, mainly used for printing statistics information
print_filter_stats <- function(df_before, df_after, step_name = "Filtering") {
  # Calculate sample sizes
  n_before <- nrow(df_before)
  n_after <- nrow(df_after)
  n_filtered <- n_before - n_after
  
  # Calculate filtering ratio (avoid division by zero)
  filter_ratio <- ifelse(n_before == 0, 0, n_filtered / n_before * 100)
  
  # Format and print output
  message(sprintf("%s before: %d", step_name, n_before))
  message(sprintf("%s after: %d", step_name, n_after))
  message(sprintf("%s: %d (proportion: %.2f%%)", 
                  step_name, 
                  n_filtered, 
                  filter_ratio))
}

## Step 1: Filter out communication events with P-value > 0.05

In [44]:
inter_single1 = inter_single[which(rowMaxs(as.matrix(inter_single[,c(20:23)]), na.rm = TRUE) < 0.05),]

In [45]:
print_filter_stats(inter_single, inter_single1, step_name = "Step1过滤")

Step1过滤前: 47251372

Step1过滤后: 10652228

Step1过滤了: 36599144 (占比: 77.46%)



## Step 2: Filter out communication events with missing values

In [46]:
inter_single1$missing_value = ifelse(
  is.na(inter_single1$`pval_Young`) | 
  is.na(inter_single1$`pval_Middle_age`) | 
  is.na(inter_single1$`pval_Old`) | 
  is.na(inter_single1$`pval_Exceptionally_old`),
  "TRUE", "FALSE"
)

In [47]:
table(inter_single1$missing_value)


  FALSE    TRUE 
5210532 5441696 

In [48]:
inter_single2 = subset(inter_single1, missing_value == "FALSE")

In [49]:
print_filter_stats(inter_single1, inter_single2, step_name = "Step2过滤")

Step2过滤前: 10652228

Step2过滤后: 5210532

Step2过滤了: 5441696 (占比: 51.09%)



## Step 3: Filter out low-probability cell-cell communication events

In [50]:
mean_probs <- rowMeans2(as.matrix(inter_single2[, 16:19]))
sorted_means <- sort(mean_probs, decreasing = TRUE)
message("95%位置的平均概率: ", sorted_means[as.integer(length(sorted_means) * 0.95)])
message("50%位置的平均概率: ", sorted_means[as.integer(length(sorted_means) * 0.5)])

95%位置的平均概率: 7.5363644210933e-05

50%位置的平均概率: 0.00290820116294407



In [51]:
inter_single3 = inter_single2[which(rowMeans2(as.matrix(inter_single2[,16:19])) > 0.002),]

In [52]:
print_filter_stats(inter_single2, inter_single3, step_name = "Step3过滤")

Step3过滤前: 5210532

Step3过滤后: 3033826

Step3过滤了: 2176706 (占比: 41.78%)



# Identify aging-related ligand-receptor pairs
***
The core function of this code is to identify aging-related ligand-receptor pairs (ARLR: Aging Related Ligand and Receptor) in cell-cell interaction (CCI) data, classify them by different aging stages (pDEG, Early, etc.), and finally generate a labeled dataset.

In [ ]:
deg <- read.csv("~/Revised_DEG_all_filter.csv")
deg <- deg[,c(-1,-2,-3)]
colnames(deg)[4:8] <- c("fdr", "celltype", "trend", "region", "stage")
deg$label <- paste0(deg$subtype,"_", deg$region)
deg <- deg[deg$subtype != "VS",]

In [57]:
head(deg,2)

,gene,p,coef,fdr,celltype,trend,region,stage,subtype,label
,<chr>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,SYT1,4.766793e-32,0.01033095,3.163244e-28,MGE_UNC5B_Inh,Up,PFC,pDEG,MGE UNC5B+ Inh,MGE UNC5B+ Inh_PFC
2,DDX17,4.082400e-31,0.03960581,1.354540e-27,MGE_UNC5B_Inh,Up,PFC,pDEG,MGE UNC5B+ Inh,MGE UNC5B+ Inh_PFC


In [67]:
pdeg <- deg[deg$stage == "pDEG", ]
Early <- deg[deg$stage == "Early", ]
Late <- deg[deg$stage == "Late", ]
Very_Late <- deg[deg$stage == "Very_Late", ]

In [69]:
inter_single3$source_label <- paste0(inter_single3$source, "_", inter_single3$region) 
inter_single3$target_label <- paste0(inter_single3$target, "_", inter_single3$region)

In [70]:
deg_label <- unique(deg$label)
cci_label <- unique(c(inter_single3$source_label, inter_single3$target_label))

In [72]:
#去除DEG中没有的细胞类型
inter_single3$TF <- ifelse(inter_single3$source_label %in% outcelltype | inter_single3$target_label %in% outcelltype, "T", "F")
inter_single4 <- subset(inter_single3, TF == "F")[,-27]

In [ ]:
# Annotate aging-related ligands and receptors by stage
annotate_aging_lr <- function(cci_data, deg_data_list) {
  cci_list <- list()
  
  for (stage in names(deg_data_list)) {
    cci_tmp <- cci_data
    cci_tmp$ligand_age   <- as.character(NA)
    cci_tmp$receptor_age <- as.character(NA)
    deg <- deg_data_list[[stage]]
    
    # Annotate ligand_age
    for (f in unique(cci_tmp$source_label)) {
      genes <- deg$gene[deg$label == f]
      idx <- cci_tmp$source_label == f
      cci_tmp$ligand_age[idx] <- ifelse(cci_tmp$ligand_gene[idx] %in% genes, cci_tmp$ligand_gene[idx], NA)
    }
    
    # Annotate receptor_age
    for (f in unique(cci_tmp$target_label)) {
      genes <- deg$gene[deg$label == f]
      idx <- cci_tmp$target_label == f
      cci_tmp$receptor_age[idx] <- ifelse(cci_tmp$receptor_gene[idx] %in% genes, cci_tmp$receptor_gene[idx], NA)
    }
    
    cci_list[[stage]] <- cci_tmp
  }
  
  return(cci_list)
}

# Run annotation for all stages
deg_list <- list(
  pDEG     = pdeg,
  Early    = Early,
  Late     = Late,
  Very_Late = Very_Late
)

CCI_list <- annotate_aging_lr(inter_single4, deg_list)

In [ ]:
# Add aging stage label to each dataset
for (stage in names(CCI_list)) {
  CCI_list[[stage]]$stage <- stage
}

# Merge data across all aging stages
merge_all <- Reduce(rbind, CCI_list)

# Flag aging-related ligand-receptor pairs (ARLR)
# ARLR = TRUE if either ligand_age or receptor_age is not NA
merge_all$ARLR <- ifelse(is.na(merge_all$ligand_age) & is.na(merge_all$receptor_age), FALSE, TRUE)

# Filter ARLR = TRUE and keep unique entries by ID
merge_all_ARLR <- merge_all[merge_all$ARLR, ]
merge_all_un <- merge_all_ARLR %>% distinct(ID, .keep_all = TRUE)

In [ ]:
# Define output directory (centralized path for easy maintenance)
output_dir <- "~/01.Project/NHPABC/Figure5/04.NHPABC_Aging_CCI/02.result/"

# Save all results
fwrite(
  x = merge_all,
  file = paste0(output_dir, "03.NHPABC_CellChat_truncatedMean_final_all.csv"),
  nThread = 120
)

# Save ARLR results (with duplicates from different DEGs)
fwrite(
  x = merge_all_ARLR,
  file = paste0(output_dir, "04.NHPABC_CellChat_truncatedMean_final_ARLR.csv"),
  nThread = 120
)

# Save unique ARLR results (duplicates removed)
fwrite(
  x = merge_all_un,
  file = paste0(output_dir, "04.NHPABC_CellChat_truncatedMean_final_ARLR_unique.csv"),
  nThread = 120
)